In [8]:
import pandas as pd 
import numpy as np
import random
import matplotlib.pyplot as plt
from vnstock import Listing

In [9]:
pool_lst_v2 = [
"FPT","CMG","ELC",#tech
"VGI","DCL",#others
"SSI","VPB","TCB","VCB","VND","TPB","BID","CTG",#fin
"VRE","VIC","VHM","CEO","DXG",#fin
"MWG","VJC","TNG","VGT","TCM","PNJ","MSH",#goods
"HAG","HNG","MSN","DBC","VNM","VHC","ANV",#goods
"GEX","VCG","CII","VSC","HUT","HBC",#industrials
"HPG","GVR","NKG","DPM","HSG",#industrials
"PVD","PVS","BSR",#industrials
"POW"#industrials
]
len(pool_lst_v2)
#remove BAF and KHG because of missing value
#remove HVN due to low ROE

47

In [144]:
import numpy as np
def scale(data):
    # if np.sum(data) == 0:
    #     return data
    if np.sum(data)==0:
        return data+1/len(data)
        #return data
    else:
        return data/np.sum(data)
    
def weight_generator(df):
    close = df
    close = close.drop(columns='time')
    close = close[pool_lst_v2]
    weight_lst = []
    for index,row in close.iterrows():
        weight_lst.append(scale(row))
    weight_df_close = pd.DataFrame(data=weight_lst)
    return weight_df_close
def weight_generator_without_time(df):
    close = df
    # close = close.drop(columns='time')
    # close = close[pool_lst_v1]
    weight_lst = []
    for index,row in close.iterrows():

        weight_lst.append(scale(row))
    weight_df_close = pd.DataFrame(data=weight_lst)
    weight_df_close=weight_df_close.clip(lower=0).fillna(0)
    return weight_df_close
def add_time_index(df,timerange):
    df['time'] = timerange
    df = df[['time'] + [col for col in df.columns if col != 'time']]
    return df
def weight_check(weight_df):
    weight_df_sum = np.sum(weight_df[weight_df.columns.values[1:]],axis=1)
    for i in weight_df_sum:
        if round(i,4) > 1:
            return "weight sum >1, ",i

            break
    if (weight_df[weight_df.columns.values[1:]].values < 0).any() == True:
        return "negative value in weight"
    else:
        return True

--------------- sector neu



In [45]:
execution_data = pd.read_csv("execution_data/pb.csv",index_col=0)
cols_name = execution_data.columns.values[1:]
execution_data[cols_name] = 1/execution_data[cols_name]

#execution_data = cashbyliabilities
execution_data_tech = execution_data[["FPT","CMG","ELC"]]
execution_data_finance = execution_data[["SSI","VPB","TCB","VCB","VND","TPB","BID","CTG","VRE","VIC","VHM","CEO","DXG"]]
execution_data_goods = execution_data[["MWG","VJC","TNG","VGT","TCM","PNJ","MSH","HAG","HNG","MSN","DBC","VNM","VHC","ANV","DCL"]]
execution_data_industrials = execution_data[["GEX","VCG","CII","VSC","HUT","HBC","HPG","GVR","NKG","DPM","HSG","PVD","PVS","BSR","POW","VGI"]]

execution_data_tech_scaled = weight_generator_without_time(execution_data_tech)
execution_data_finance_scaled = weight_generator_without_time(execution_data_finance)
execution_data_goods_scaled = weight_generator_without_time(execution_data_goods)
execution_data_industrials_scaled = weight_generator_without_time(execution_data_industrials)


In [46]:
global_weight = pd.concat([
    execution_data_tech_scaled,
    execution_data_finance_scaled,
    execution_data_goods_scaled,
    execution_data_industrials_scaled
    
],axis=1)
global_weight_scaled = weight_generator_without_time(global_weight)
tmp_df = pd.read_csv("execution_data/roe.csv")
tmp_df_date = tmp_df['time']
timerange = np.array(tmp_df_date)
global_weight_scaled['time'] = timerange
global_weight_scaled = global_weight_scaled[['time'] + [col for col in global_weight_scaled.columns if col != 'time']]

In [47]:
global_weight_scaled

,time,FPT,CMG,ELC,SSI,VPB,TCB,VCB,VND,TPB,...,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW,VGI
0,2020-01-05,0.024812,0.038859,0.186329,0.026049,0.018506,0.021985,0.006510,0.033067,0.018146,...,0.005866,0.010432,0.026304,0.013163,0.010927,0.033716,0.019874,0.013401,0.011478,0.003589
1,2020-01-12,0.024812,0.038859,0.186329,0.026049,0.018506,0.021985,0.006510,0.033067,0.018146,...,0.005866,0.010432,0.026304,0.013163,0.010927,0.033716,0.019874,0.013401,0.011478,0.003589
2,2020-01-19,0.024812,0.038859,0.186329,0.026049,0.018506,0.021985,0.006510,0.033067,0.018146,...,0.005866,0.010432,0.026304,0.013163,0.010927,0.033716,0.019874,0.013401,0.011478,0.003589
3,2020-01-26,0.024812,0.038859,0.186329,0.026049,0.018506,0.021985,0.006510,0.033067,0.018146,...,0.005866,0.010432,0.026304,0.013163,0.010927,0.033716,0.019874,0.013401,0.011478,0.003589
4,2020-02-02,0.024812,0.038859,0.186329,0.026049,0.018506,0.021985,0.006510,0.033067,0.018146,...,0.005866,0.010432,0.026304,0.013163,0.010927,0.033716,0.019874,0.013401,0.011478,0.003589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,2025-10-05,0.046851,0.079993,0.123156,0.014788,0.022877,0.022342,0.015128,0.022791,0.029295,...,0.013085,0.010676,0.022777,0.014603,0.023264,0.030058,0.019571,0.014724,0.021465,0.003345
301,2025-10-12,0.046851,0.079993,0.123156,0.014788,0.022877,0.022342,0.015128,0.022791,0.029295,...,0.013085,0.010676,0.022777,0.014603,0.023264,0.030058,0.019571,0.014724,0.021465,0.003345
302,2025-10-19,0.046851,0.079993,0.123156,0.014788,0.022877,0.022342,0.015128,0.022791,0.029295,...,0.013085,0.010676,0.022777,0.014603,0.023264,0.030058,0.019571,0.014724,0.021465,0.003345
303,2025-10-26,0.046851,0.079993,0.123156,0.014788,0.022877,0.022342,0.015128,0.022791,0.029295,...,0.013085,0.010676,0.022777,0.014603,0.023264,0.030058,0.019571,0.014724,0.021465,0.003345


In [49]:
global_weight_scaled.to_csv("weight/low_pb_sec_neu.csv")

----------------------------- Normal

In [161]:
execution_data = pd.read_csv("execution_data/roe.csv",index_col=0)
execution_data

,time,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,2020-01-05,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,-0.002984,0.070153
1,2020-01-12,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,-0.002984,0.070153
2,2020-01-19,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,-0.002984,0.070153
3,2020-01-26,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,-0.002984,0.070153
4,2020-02-02,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,-0.002984,0.070153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2025-10-12,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
302,2025-10-19,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
303,2025-10-26,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
304,2025-11-02,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115


In [162]:
cols_name = execution_data.columns.values[1:]
execution_data[cols_name] = execution_data[cols_name].clip(0)
execution_data 

,time,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,2020-01-05,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,0.000000,0.070153
1,2020-01-12,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,0.000000,0.070153
2,2020-01-19,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,0.000000,0.070153
3,2020-01-26,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,0.000000,0.070153
4,2020-02-02,0.195676,0.071317,0.039956,0.079643,0.114733,0.072910,0.221472,0.172719,0.218914,...,0.086647,0.169906,0.066403,0.063016,0.053351,0.113738,0.021467,0.036743,0.000000,0.070153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2025-10-12,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
302,2025-10-19,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
303,2025-10-26,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115
304,2025-11-02,0.273707,0.130680,0.115982,0.255022,0.023207,0.130601,0.135363,0.136738,0.167670,...,0.109701,0.120224,0.113152,0.031463,0.073454,0.065082,0.054572,0.104111,0.036705,0.058115


In [163]:
weight = weight_generator(execution_data)
weight = weight
timerange = execution_data['time']
weight=add_time_index(weight,timerange)

In [165]:
weight_check(weight)
# weight_df_sum = np.sum(weight[weight.columns.values[1:]],axis=1)
# weight_df_sum

True

In [166]:
weight.to_csv("weight/roe.csv")

______________________ equal weight

In [ ]:
pool_lst_v2 = [
"FPT","CMG","ELC",#tech
"VGI","DCL",#others
"SSI","VPB","TCB","VCB","VND","TPB","BID","CTG",#fin
"VRE","VIC","VHM","CEO","DXG",#fin
"MWG","VJC","HVN","TNG","VGT","TCM","PNJ","MSH",#goods
"HAG","HNG","MSN","DBC","VNM","VHC","ANV",#goods
"GEX","VCG","CII","VSC","HUT","HBC",#industrials
"HPG","GVR","NKG","DPM","HSG",#industrials
"PVD","PVS","BSR",#industrials
"POW"#industrials
]

In [11]:
tmp_df = pd.read_csv("raw_data/pv/FPT.csv")
tmp_df_date = tmp_df['time']
timerange = np.array(tmp_df_date)
weight_df = pd.DataFrame(data=timerange)
for i1 in pool_lst_v2:
    weight_df[i1] = np.nan
weight_df[["FPT","CMG","ELC"]] = weight_df[["FPT","CMG","ELC"]].fillna(0.2/3)
weight_df[["SSI","VPB","TCB","VCB","VND","TPB","BID","CTG","VRE","VIC","VHM","CEO","DXG"]] = weight_df[["SSI","VPB","TCB","VCB","VND","TPB","BID","CTG","VRE","VIC","VHM","CEO","DXG"]].fillna(0.2/13)
weight_df[["MWG","VJC","HVN","TNG","VGT","TCM","PNJ","MSH","HAG","HNG","MSN","DBC","VNM","VHC","ANV"]] = weight_df[["MWG","VJC","HVN","TNG","VGT","TCM","PNJ","MSH","HAG","HNG","MSN","DBC","VNM","VHC","ANV"]].fillna(0.2/15)
weight_df[["GEX","VCG","CII","VSC","HUT","HBC","HPG","GVR","NKG","DPM","HSG","PVD","PVS","BSR","POW"]] = weight_df[["GEX","VCG","CII","VSC","HUT","HBC","HPG","GVR","NKG","DPM","HSG","PVD","PVS","BSR","POW"]].fillna(0.2/15)
weight_df[["VGI","DCL"]] = weight_df[["VGI","DCL"]].fillna(0.2/2)

weight_df

,0,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,2020-01-05,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
1,2020-01-12,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
2,2020-01-19,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
3,2020-01-26,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
4,2020-02-02,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,2025-10-05,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
301,2025-10-12,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
302,2025-10-19,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333
303,2025-10-26,0.066667,0.066667,0.066667,0.1,0.1,0.015385,0.015385,0.015385,0.015385,...,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333,0.013333


In [12]:
weight_df.to_csv("weight/equal_weight_sector_neu.csv")

------------------------- delta

In [155]:
close_df = pd.read_csv("execution_data/close.csv",index_col=0)
# close_df=close_df.drop(columns="HVN")
close_df_date = close_df['time']
timerange = np.array(close_df_date)

In [156]:
close_delta_df = close_df[list(close_df.columns.values)[1:]].pct_change(1)
close_delta_df=close_delta_df.clip(lower=0)
#close_delta_df
close_delta_df

,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,VND,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000000,0.000000,0.083871,0.000000,0.000000,0.003040,0.031532,0.000000,0.000000,0.000000,...,0.026667,0.001200,0.000000,0.180412,0.000000,0.013233,0.000000,0.017083,0.000000,0.000000
2,0.001906,0.000000,0.000000,0.043288,0.012195,0.051515,0.040757,0.010055,0.055627,0.010791,...,0.000000,0.033573,0.018072,0.106987,0.000000,0.044776,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.003049,0.103940,0.004016,0.002882,0.076923,0.034389,0.000000,0.046263,...,0.025510,0.052204,0.061144,0.000000,0.007059,0.062500,0.021198,0.021912,0.148325,0.028155
4,0.000000,0.000000,0.006079,0.000000,0.040000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,0.028908,0.021711,0.000000,0.020802,0.053571,0.081009,0.088136,0.026076,0.040519,0.075000,...,0.000000,0.070524,0.050562,0.071856,0.016632,0.052632,0.038462,0.041270,0.120893,0.039146
302,0.000000,0.000000,0.000000,0.000000,0.000000,0.002457,0.000000,0.033037,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
303,0.108967,0.038874,0.021687,0.085758,0.050847,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.007500,0.026316,0.000000,0.000000
304,0.063460,0.081290,0.035377,0.119887,0.062903,0.000000,0.000000,0.000000,0.001681,0.000000,...,0.063492,0.011364,0.094162,0.032258,0.043478,0.027607,0.074442,0.051282,0.000000,0.070896


In [ ]:
close_delta_weight = weight_generator_without_time(close_delta_df)
close_delta_weight['time'] = timerange
close_delta_weight = close_delta_weight[['time'] + [col for col in close_delta_weight.columns if col != 'time']]
#weight_sum_check(close_delta_weight)
#close_delta_weight.to_csv("weight/close_delta1w_clipneg.csv")

#(close_delta_weight[close_delta_weight.columns.values[1:]].values < 0).any()

In [159]:
weight_check(close_delta_weight)


0      0.0
1      1.0
2      1.0
3      1.0
4      1.0
      ... 
301    1.0
302    1.0
303    1.0
304    1.0
305    1.0
Length: 306, dtype: float64

--------------------- delta index


In [145]:
close_df = pd.read_csv("execution_data/close.csv",index_col=0)
# close_df=close_df.drop(columns="HVN")
close_df_date = close_df['time']
timerange = np.array(close_df_date)

In [146]:
close_delta_df = close_df[list(close_df.columns.values)[1:]].pct_change(1)
close_delta_df=close_delta_df.clip(lower=0)
#close_delta_df
close_delta_df

,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,VND,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000000,0.000000,0.083871,0.000000,0.000000,0.003040,0.031532,0.000000,0.000000,0.000000,...,0.026667,0.001200,0.000000,0.180412,0.000000,0.013233,0.000000,0.017083,0.000000,0.000000
2,0.001906,0.000000,0.000000,0.043288,0.012195,0.051515,0.040757,0.010055,0.055627,0.010791,...,0.000000,0.033573,0.018072,0.106987,0.000000,0.044776,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.003049,0.103940,0.004016,0.002882,0.076923,0.034389,0.000000,0.046263,...,0.025510,0.052204,0.061144,0.000000,0.007059,0.062500,0.021198,0.021912,0.148325,0.028155
4,0.000000,0.000000,0.006079,0.000000,0.040000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,0.028908,0.021711,0.000000,0.020802,0.053571,0.081009,0.088136,0.026076,0.040519,0.075000,...,0.000000,0.070524,0.050562,0.071856,0.016632,0.052632,0.038462,0.041270,0.120893,0.039146
302,0.000000,0.000000,0.000000,0.000000,0.000000,0.002457,0.000000,0.033037,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
303,0.108967,0.038874,0.021687,0.085758,0.050847,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.007500,0.026316,0.000000,0.000000
304,0.063460,0.081290,0.035377,0.119887,0.062903,0.000000,0.000000,0.000000,0.001681,0.000000,...,0.063492,0.011364,0.094162,0.032258,0.043478,0.027607,0.074442,0.051282,0.000000,0.070896


In [147]:
vnindex_df = pd.read_csv("raw_data/macro/vnindex.csv",index_col=0)
vnindex_df=vnindex_df.set_index('time')
vnindex_df = vnindex_df["2020-01-01":]
vnindex_df.reset_index(drop=True, inplace=True)
vnindex_df = vnindex_df['close']
#vnindex_df
vnindex_delta_df = vnindex_df.pct_change(1)


The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.


In [148]:
exceesess_delta = close_delta_df.sub(vnindex_delta_df, axis=0)
exceesess_delta = exceesess_delta.clip(0)
exceesess_delta

,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,VND,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000000,0.000000,0.080348,0.000000,0.000000,0.000000,0.028009,0.000000,0.000000,0.000000,...,0.023144,0.000000,0.000000,0.176890,0.000000,0.009710,0.000000,0.013560,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.032530,0.001437,0.040757,0.029998,0.000000,0.044869,0.000033,...,0.000000,0.022815,0.007314,0.096228,0.000000,0.034018,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.091171,0.000000,0.000000,0.064154,0.021620,0.000000,0.033495,...,0.012742,0.039436,0.048375,0.000000,0.000000,0.049731,0.008430,0.009144,0.135557,0.015387
4,0.055312,0.055312,0.061391,0.055312,0.095312,0.055312,0.055312,0.055312,0.055312,0.055312,...,0.055312,0.055312,0.055312,0.055312,0.055312,0.055312,0.055312,0.055312,0.055312,0.055312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,0.000000,0.000000,0.000000,0.000000,0.000000,0.019198,0.026324,0.000000,0.000000,0.013189,...,0.000000,0.008713,0.000000,0.010045,0.000000,0.000000,0.000000,0.000000,0.059082,0.000000
302,0.009362,0.009362,0.009362,0.009362,0.009362,0.011819,0.009362,0.042399,0.009362,0.009362,...,0.009362,0.009362,0.009362,0.009362,0.009362,0.009362,0.009362,0.009362,0.009362,0.009362
303,0.136699,0.066606,0.049419,0.113490,0.078580,0.027732,0.027732,0.027732,0.027732,0.027732,...,0.027732,0.027732,0.027732,0.027732,0.027732,0.027732,0.035232,0.054048,0.027732,0.027732
304,0.089321,0.107152,0.061239,0.145749,0.088765,0.025862,0.025862,0.025862,0.027542,0.025862,...,0.089354,0.037225,0.120024,0.058120,0.069340,0.053469,0.100303,0.077144,0.025862,0.096757


In [149]:
exceesess_delta = weight_generator_without_time(exceesess_delta)
exceesess_delta['time'] = timerange
exceesess_delta = exceesess_delta[['time'] + [col for col in exceesess_delta.columns if col != 'time']]
#close_delta_weight.to_csv("weight/statistical_close_delta1w_clipneg.csv")

In [151]:
exceesess_delta

,time,FPT,CMG,ELC,VGI,DCL,SSI,VPB,TCB,VCB,...,HBC,HPG,GVR,NKG,DPM,HSG,PVD,PVS,BSR,POW
0,2020-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2020-01-12,0.000000,0.000000,0.103712,0.000000,0.000000,0.000000,0.036153,0.000000,0.000000,...,0.029874,0.000000,0.000000,0.228325,0.000000,0.012533,0.000000,0.017503,0.000000,0.000000
2,2020-01-19,0.000000,0.000000,0.000000,0.063826,0.002819,0.079968,0.058860,0.000000,0.088037,...,0.000000,0.044765,0.014350,0.188809,0.000000,0.066746,0.000000,0.000000,0.000000,0.000000
3,2020-01-26,0.000000,0.000000,0.000000,0.098611,0.000000,0.000000,0.069390,0.023385,0.000000,...,0.013781,0.042654,0.052323,0.000000,0.000000,0.053790,0.009117,0.009890,0.146619,0.016642
4,2020-02-02,0.020906,0.020906,0.023204,0.020906,0.036025,0.020906,0.020906,0.020906,0.020906,...,0.020906,0.020906,0.020906,0.020906,0.020906,0.020906,0.020906,0.020906,0.020906,0.020906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2025-10-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.030630,0.042000,0.000000,0.000000,...,0.000000,0.013902,0.000000,0.016027,0.000000,0.000000,0.000000,0.000000,0.094263,0.000000
302,2025-10-19,0.006093,0.006093,0.006093,0.006093,0.006093,0.007692,0.006093,0.027593,0.006093,...,0.006093,0.006093,0.006093,0.006093,0.006093,0.006093,0.006093,0.006093,0.006093,0.006093
303,2025-10-26,0.075072,0.036578,0.027140,0.062326,0.043154,0.015230,0.015230,0.015230,0.015230,...,0.015230,0.015230,0.015230,0.015230,0.015230,0.015230,0.019349,0.029682,0.015230,0.015230
304,2025-11-02,0.032687,0.039212,0.022410,0.053337,0.032483,0.009464,0.009464,0.009464,0.010079,...,0.032699,0.013623,0.043923,0.021269,0.025375,0.019567,0.036706,0.028231,0.009464,0.035408


In [153]:
#weight_sum_check_df = np.sum(exceesess_delta[exceesess_delta.columns.values[1:]],axis=1)
weight_check(exceesess_delta)

True

In [154]:
exceesess_delta.to_csv("weight/excessive_delta1.csv")